# NOURA EL KHOLTI

## Arbre De Decision ID3 (Iterative Dichotomiser 3)

L'objectif est de construire un arbre de décision capable de classer des données en fonction de caractéristiques catégorielles.

### 1. Préparation des Données

Nous utilisons un jeu de données représentant le comportement d’achat des consommateurs afin de déterminer quel modèle de smartphone est le plus susceptible d’être choisi.

Les attributs pris en compte sont:
- Marque: Positionnement (Premium, Milieu de gamme).
- Batterie: Autonomie constatée (Excellente, Moyenne).
- Prix: Rapport au budget (Élevé, Abordable).
- Achat: Variable cible (Oui / Non).

In [ ]:
import pandas as pd
import numpy as np
import math

# Préparation du dataset : Décision d'achat Smartphone
donnees = {
    'marque': ['Premium', 'Premium', 'Premium', 'Standard', 'Standard', 'Standard', 'Premium', 'Standard'],
    'batterie': ['Excellente', 'Excellente', 'Moyenne', 'Excellente', 'Moyenne', 'Moyenne', 'Moyenne', 'Excellente'],
    'prix': ['Élevé', 'Abordable', 'Élevé', 'Abordable', 'Abordable', 'Élevé', 'Abordable', 'Élevé'],
    'achat': ['Oui', 'Oui', 'Non', 'Oui', 'Oui', 'Non', 'Oui', 'Non']
}

df = pd.DataFrame(donnees)

### 2. Socle Mathématique

L'algorithme ID3 utilise des mesures statistiques pour construire l'arbre de manière optimale.

- Entropie de Shannon: L'entropie $H(S)$ quantifie le désordre dans l'ensemble de données. Si toutes les décisions sont identiques, l'entropie est nulle.
$$H(S) = - \sum_{i=1}^{n} p_i \log_2(p_i)$$

- Gain d'Information: Le gain d'information $G(S, A)$ mesure l'efficacité d'un attribut $A$ à séparer les données selon la cible. L'algorithme choisit toujours l'attribut avec le gain le plus élevé pour créer un nœud.
$$Gain(S, A) = H(S) - \sum_{v \in \text{Valeurs}(A)} \frac{|S_v|}{|S|} H(S_v)$$

### 3. Implémentation de l'Algorithme

Fonctions de Calcul

In [ ]:
def calcul_entropie(colonne_cible):
    elements, comptes = np.unique(colonne_cible, return_counts=True)
    entropie = np.sum([(-comptes[i]/np.sum(comptes)) * math.log2(comptes[i]/np.sum(comptes)) 
                       for i in range(len(elements))])
    return entropie

def gain_information(donnees, nom_attribut, nom_cible):
    entropie_totale = calcul_entropie(donnees[nom_cible])
    valeurs, comptes = np.unique(donnees[nom_attribut], return_counts=True)
    
    entropie_ponderee = np.sum([(comptes[i]/np.sum(comptes)) * calcul_entropie(donnees[donnees[nom_attribut] == valeurs[i]][nom_cible]) 
                                 for i in range(len(valeurs))])
    
    return entropie_totale - entropie_ponderee

L'algorithme parcourt les données et divise les nœuds jusqu'à obtenir des feuilles "pures" ou épuiser les attributs.

In [ ]:
def construire_arbre(donnees, attributs, nom_cible):
    # Si toutes les cibles sont identiques, on retourne la classe
    if len(np.unique(donnees[nom_cible])) <= 1:
        return np.unique(donnees[nom_cible])[0]
    
    # Si plus d'attributs, on retourne la classe majoritaire
    if len(attributs) == 0:
        return donnees[nom_cible].mode()[0]
    
    # Sélection du meilleur attribut (Max Gain)
    gains = [gain_information(donnees, a, nom_cible) for a in attributs]
    meilleur_attr = attributs[np.argmax(gains)]
    
    arbre = {meilleur_attr: {}}
    nouv_attributs = [a for a in attributs if a != meilleur_attr]
    
    for valeur in np.unique(donnees[meilleur_attr]):
        sous_ensemble = donnees[donnees[meilleur_attr] == valeur]
        arbre[meilleur_attr][valeur] = construire_arbre(sous_ensemble, nouv_attributs, nom_cible)
        
    return arbre

### 4. Test et Validation

En exécutant le code sur notre jeu de données "Smartphone", nous obtenons la structure logique suivante:

In [ ]:
attributs_cles = ["marque", "batterie", "prix"]
mon_arbre = construire_arbre(df, attributs_cles, "achat")

import pprint
pprint.pprint(mon_arbre)

```
{
    "prix": {
        "Abordable": "Oui",
        "Élevé": {
            "marque": {
                "Premium": {
                    "batterie": {
                        "Excellente": "Oui",
                        "Moyenne": "Non"
                    }
                },
                "Standard": "Non"
            }
        }
    }
}
```

L'arbre pourrait par exemple décider que si le Prix est Abordable, l'achat est systématique (Oui), mais si le Prix est Élevé, il vérifie alors la Batterie ou la Marque.